# GAS-BayesSHAP — matched ShaplEIG vs GAS (unique-query comparison)

Closes audit P1-7: the earlier summary compared ShaplEIG at 64/128/256 **unique** queries against GAS at nominal K=128/256/512 (~393/480/565 actual unique evals) — not actually matched.  This runs both methods at the **same nominal budgets** and reports each method's *actual unique* coalition evaluations side by side (`matched_unique` is True only when the counts agree within 35%).  Orchestrates `scripts/run_matched_shaplEIG.py` only.

## 0. Environment & config

In [ ]:
import sys, os, time, subprocess
from pathlib import Path
sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
print("GAS-BayesSHAP", gas_bayesshap.__version__)

N_INST  = int(os.environ.get("N_INST", "2"))       # instances per dataset
BUDGETS = os.environ.get("BUDGETS", "64,128,256")
SKIP    = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        err = (r.stderr or r.stdout or '').strip().splitlines()
        print(' | '.join(err[-6:]) if err else '<no output>')
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N_INST={N_INST} BUDGETS={BUDGETS} SKIP={sorted(SKIP)}")

## A. Matched comparison (wine + air)

Runs GAS (spec range) and the ShaplEIG port at nominal K ∈ {64,128,256} on N_INST instances per dataset, recording actual unique coalition evals for both.  **~30–45 min on a laptop** (12 ShaplEIG runs, each refits a GP per acquisition round).  Smoke: `N_INST=1 BUDGETS=64`.

In [ ]:
run("run_matched_shaplEIG.py", "--n", str(N_INST), "--budgets", BUDGETS,
    tag=f"A. matched ShaplEIG vs GAS (N={N_INST}, K={BUDGETS})", skip=False)

## B. Summary table

In [ ]:
import pandas as pd
p = ROOT / "main_results" / "paper_matched_shaplEIG_comparison.csv"
if not p.exists():
    print("NOT FOUND — run section A first")
else:
    d = pd.read_csv(p)
    cols = ["dataset", "instance", "nominal_K", "gas_rmse",
            "shaplEIG_rmse", "gas_unique_evals", "shaplEIG_unique_queries",
            "matched_unique"]
    print(d[cols].to_string(index=False))
    g = d.groupby(["dataset", "nominal_K"]).agg(
        gas_rmse=("gas_rmse", "mean"), gas_unique=("gas_unique_evals", "mean"),
        shaplEIG_rmse=("shaplEIG_rmse", "mean"),
        shaplEIG_unique=("shaplEIG_unique_queries", "mean"),
    ).round(5)
    print("\n=== mean over instances ===")
    print(g.to_string())
    print("\nHONEST NOTE: rows with matched_unique=False are NOT a matched"
          " unique-query comparison — report the actual counts, not a claim.")

## Notes
- The ShaplEIG port is pure NumPy/SciPy (same math as slds-lmu/shapleig@
  d52c09e; no torch — macOS crash-free).  GAS uses the spec range.
- Commit `paper_matched_shaplEIG_comparison.csv` when done.